<a href="https://colab.research.google.com/github/pablobelmiro/olist_exploration/blob/main/01_intro_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olist E-Commerce: Capítulo 1, Modelo Relacional + EDA

Antes de rodar: sobe os 9 CSVs do dataset (`olist_orders_dataset.csv`, `olist_order_items_dataset.csv`, `olist_order_payments_dataset.csv`, `olist_order_reviews_dataset.csv`, `olist_customers_dataset.csv`, `olist_sellers_dataset.csv`, `olist_products_dataset.csv`, `olist_geolocation_dataset.csv`, `product_category_name_translation.csv`) pro mesmo diretório desse notebook no Colab (painel de Arquivos à esquerda, ou `files.upload()` na célula abaixo).

Não precisa de GPU (T4) pra esse capítulo, é só pandas.

In [ ]:
import pandas as pd
import json

pd.set_option('display.max_columns', None)

## 1. Carregando os 9 CSVs

In [ ]:
orders = pd.read_csv('olist_orders_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
geolocation = pd.read_csv('olist_geolocation_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

tabelas = {
    'orders': orders,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'customers': customers,
    'sellers': sellers,
    'products': products,
    'geolocation': geolocation,
    'category_translation': category_translation,
}

for nome, df in tabelas.items():
    print(f"{nome}: {df.shape[0]} linhas, {df.shape[1]} colunas")

orders: 99441 linhas, 8 colunas
order_items: 112650 linhas, 7 colunas
payments: 103886 linhas, 5 colunas
reviews: 99224 linhas, 7 colunas
customers: 99441 linhas, 5 colunas
sellers: 3095 linhas, 4 colunas
products: 32951 linhas, 9 colunas
geolocation: 1000163 linhas, 5 colunas
category_translation: 71 linhas, 2 colunas


## 2. Qualidade dos dados: nulos por coluna

Só nas tabelas onde nulo é esperado/interessante (datas de entrega ausentes, comentário de review vazio, etc).

In [ ]:
for nome, df in tabelas.items():
    nulos = df.isnull().sum()
    nulos = nulos[nulos > 0]
    if len(nulos):
        print(f"--- {nome} ---")
        for col, qtd in nulos.items():
            pct = 100 * qtd / len(df)
            print(f"  {col}: {qtd} nulos ({pct:.1f}%)")

--- orders ---
  order_approved_at: 160 nulos (0.2%)
  order_delivered_carrier_date: 1783 nulos (1.8%)
  order_delivered_customer_date: 2965 nulos (3.0%)
--- reviews ---
  review_comment_title: 87656 nulos (88.3%)
  review_comment_message: 58247 nulos (58.7%)
--- products ---
  product_category_name: 610 nulos (1.9%)
  product_name_lenght: 610 nulos (1.9%)
  product_description_lenght: 610 nulos (1.9%)
  product_photos_qty: 610 nulos (1.9%)
  product_weight_g: 2 nulos (0.0%)
  product_length_cm: 2 nulos (0.0%)
  product_height_cm: 2 nulos (0.0%)
  product_width_cm: 2 nulos (0.0%)


## 3. Montando o dataframe mestre

Granularidade: 1 linha = 1 item de pedido (`order_items`), porque um `order_id` pode ter múltiplos itens/vendedores. O merge com `orders`, `customers`, `payments`, `reviews`, `products` e `sellers` é feito a partir daí.

In [ ]:
produtos_com_categoria_en = products.merge(category_translation, on='product_category_name', how='left')

mestre = (
    order_items
    .merge(orders, on='order_id', how='left')
    .merge(customers, on='customer_id', how='left')
    .merge(produtos_com_categoria_en, on='product_id', how='left')
    .merge(sellers, on='seller_id', how='left')
    .merge(payments, on='order_id', how='left')
    .merge(reviews, on='order_id', how='left')
)

print(f"order_items sozinho: {len(order_items)} linhas")
print(f"dataframe mestre após os merges: {len(mestre)} linhas")
print(f"colunas: {list(mestre.columns)}")

order_items sozinho: 112650 linhas
dataframe mestre após os merges: 118310 linhas
colunas: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'review_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


Repara que o número de linhas do mestre pode ficar maior que `order_items` sozinho: cada pedido pode ter mais de um registro de pagamento (parcelamento dividido) e mais de uma review, então o merge multiplica linhas nesses casos. Isso é esperado, não é bug, vale comentar no post.

## 4. Os 4 cenários de ML que essa série de posts vai cobrir

1. **Atraso na entrega** (classificação binária): `order_delivered_customer_date` vs `order_estimated_delivery_date`.
2. **Nota da avaliação** (classificação multiclasse / regressão): `review_score`, de 1 a 5.
3. **Valor do frete ou do pedido** (regressão): `freight_value` ou soma de `price` por pedido.
4. **Segmentação de clientes** (clustering, RFM: Recência, Frequência, Valor monetário): sem target, agrupa por comportamento de compra.

Cada um puxa um subconjunto diferente de feature do mesmo dataframe mestre, capítulos futuros da série entram em cada um.

## 5. Pedidos por mês (real, pro gráfico do post)

Conta pedidos únicos (`order_id`) por mês de compra, ordenado cronologicamente. Vira o JSON que o gráfico do blog consome.

In [ ]:
orders_com_data = orders.copy()
orders_com_data['order_purchase_timestamp'] = pd.to_datetime(orders_com_data['order_purchase_timestamp'])
orders_com_data['mes'] = orders_com_data['order_purchase_timestamp'].dt.to_period('M').astype(str)

pedidos_por_mes = (
    orders_com_data
    .groupby('mes')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'pedidos'})
    .sort_values('mes')
)

print(pedidos_por_mes.to_string(index=False))

    mes  pedidos
2016-09        4
2016-10      324
2016-12        1
2017-01      800
2017-02     1780
2017-03     2682
2017-04     2404
2017-05     3700
2017-06     3245
2017-07     4026
2017-08     4331
2017-09     4285
2017-10     4631
2017-11     7544
2017-12     5673
2018-01     7269
2018-02     6728
2018-03     7211
2018-04     6939
2018-05     6873
2018-06     6167
2018-07     6292
2018-08     6512
2018-09       16
2018-10        4


In [ ]:
orders_per_month_json = [
    {'month': row['mes'], 'orders': int(row['pedidos'])}
    for _, row in pedidos_por_mes.iterrows()
]

with open('orders-per-month.json', 'w', encoding='utf-8') as f:
    json.dump(orders_per_month_json, f, ensure_ascii=False, indent=2)

print(json.dumps(orders_per_month_json, ensure_ascii=False, indent=2))

[
  {
    "month": "2016-09",
    "orders": 4
  },
  {
    "month": "2016-10",
    "orders": 324
  },
  {
    "month": "2016-12",
    "orders": 1
  },
  {
    "month": "2017-01",
    "orders": 800
  },
  {
    "month": "2017-02",
    "orders": 1780
  },
  {
    "month": "2017-03",
    "orders": 2682
  },
  {
    "month": "2017-04",
    "orders": 2404
  },
  {
    "month": "2017-05",
    "orders": 3700
  },
  {
    "month": "2017-06",
    "orders": 3245
  },
  {
    "month": "2017-07",
    "orders": 4026
  },
  {
    "month": "2017-08",
    "orders": 4331
  },
  {
    "month": "2017-09",
    "orders": 4285
  },
  {
    "month": "2017-10",
    "orders": 4631
  },
  {
    "month": "2017-11",
    "orders": 7544
  },
  {
    "month": "2017-12",
    "orders": 5673
  },
  {
    "month": "2018-01",
    "orders": 7269
  },
  {
    "month": "2018-02",
    "orders": 6728
  },
  {
    "month": "2018-03",
    "orders": 7211
  },
  {
    "month": "2018-04",
    "orders": 6939
  },
  {
    "month": "